# U-Net: Convolutional Networks for Biomedical Image Segmentation
### Full PyTorch reimplementation of Ronneberger, Fischer & Brox (2015), MICCAI

This notebook reimplements the original U-Net paper end-to-end, and is designed to run standalone in
Google Colab with **no manual dataset upload required**: it trains on the **real ISBI 2012 EM
segmentation challenge** data (Drosophila VNC) — the exact dataset the paper's Table 1 results are
reported on — fetched automatically via an anonymous `git clone`, no account or API key needed.

**What's included, matching the paper section-by-section:**

| Paper Section | Notebook Section |
|---|---|
| Fig. 1 — U-Net architecture (unpadded 3x3 convs, 572→388) | `2. Model Architecture` — `UNetOriginal` |
| Convenience "same"-padding variant (512→512) | `2. Model Architecture` — `UNetPadded` |
| §3 Training — He/Kaiming init, N = 9·64 | `weights_init_he()` |
| Eq. (2) — weight map $w(x) = w_c(x) + w_0\cdot\exp(-\frac{(d_1+d_2)^2}{2\sigma^2})$ | `3. Weighted Loss` |
| §3 — pixel-wise weighted cross-entropy, Eq. (1) | `WeightedCrossEntropyLoss` |
| §3.1 — elastic deformation augmentation, 3x3 grid, σ=10px | `4. Data Augmentation` |
| §3 — SGD, momentum 0.99, batch size 1 | `6. Training Loop` |
| Table 1 / Table 2 — IoU / pixel accuracy evaluation | `5. Metrics` |
| Fig. 2, 3, 4 — qualitative visualizations | `7. Visualization` |

> Reference: Ronneberger, O., Fischer, P., Brox, T. (2015). *U-Net: Convolutional Networks for
> Biomedical Image Segmentation*. MICCAI 2015, LNCS 9351, pp. 234–241.


## 0. Setup & Environment

In [ ]:
# ============================================================
# 0.1  Package installation (Colab-safe; skips if already present)
# ============================================================
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for _pkg, _pip in [("albumentations", "albumentations"),
                    ("cv2", "opencv-python-headless"),
                    ("scipy", "scipy")]:
    _ensure(_pkg, _pip)

print("All required packages are available.")


In [ ]:
# ============================================================
# 0.2  Imports
# ============================================================
import os
import math
import random
import time
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import cv2
from scipy.ndimage import distance_transform_edt, label as cc_label

import matplotlib.pyplot as plt

import albumentations as A

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# ============================================================
# 0.3  Device / GPU check
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. In Colab go to Runtime > Change runtime type > GPU (T4).")
print(f"Using device    : {device}")


## 1. Real Dataset — ISBI 2012 EM Segmentation Challenge (Drosophila VNC)

This is the **actual dataset the paper reports its best score on** (Table 1: warping error
0.000353, rand error 0.0382) — 30 fully-annotated 512×512 serial-section transmission electron
microscopy images of the Drosophila first-instar larva ventral nerve cord, with binary
cell/membrane ground truth.

The official challenge test-set labels are kept secret by the organizers (evaluation requires
submitting predictions to their server), so this notebook downloads the **30 labeled training
images** and creates its own train/validation split from them — matching the paper's own
"very few annotated images" regime exactly, rather than approximating it.

**No manual upload or account/API-key is required.** The data is fetched with a plain, anonymous
`git clone` from a public mirror of the dataset
([`zhixuhao/unet`](https://github.com/zhixuhao/unet), a well-known, widely used repackaging of the
official ISBI 2012 challenge release as flat PNG files) so the notebook still runs standalone.

> To use a different real dataset instead — Kaggle **2018 Data Science Bowl** (nuclei), the
> **PhC-U373** / **DIC-HeLa** cell-tracking sets, or **Oxford-IIIT Pet** — replace
> `ISBIMembraneDataset` with your own `torch.utils.data.Dataset` that returns
> `(image[H,W] float32 in [0,1], mask[H,W] int64 in {0,1})`; everything downstream (weight maps,
> augmentation, loss, training loop) is dataset-agnostic. See Section 9 for details.


In [ ]:
# ============================================================
# 1.1  Download the real ISBI EM membrane dataset (anonymous git clone, no credentials)
# ============================================================
import subprocess

DATA_ROOT = "/content/isbi_membrane_data"
REPO_DIR = os.path.join(DATA_ROOT, "unet")
IMAGE_DIR = os.path.join(REPO_DIR, "data", "membrane", "train", "image")
LABEL_DIR = os.path.join(REPO_DIR, "data", "membrane", "train", "label")


def download_isbi_membrane_dataset():
    """Fetch the 30 labeled ISBI-2012 EM training images via an anonymous git clone."""
    if os.path.isdir(IMAGE_DIR) and len(os.listdir(IMAGE_DIR)) > 0:
        print(f"Dataset already present at {IMAGE_DIR} ({len(os.listdir(IMAGE_DIR))} images).")
        return
    os.makedirs(DATA_ROOT, exist_ok=True)
    print("Downloading ISBI EM membrane dataset (zhixuhao/unet mirror)...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/zhixuhao/unet.git", REPO_DIR],
        check=True,
    )
    print(f"Done. {len(os.listdir(IMAGE_DIR))} labeled training images available.")


download_isbi_membrane_dataset()


In [ ]:
# ============================================================
# 1.2  Real dataset class
# ============================================================
class ISBIMembraneDataset(Dataset):
    """The paper's own ISBI-2012 EM segmentation dataset (Drosophila VNC), 512x512 grayscale
    images with binary cell/membrane ground truth. Returns (image float32 [H,W] in [0,1],
    mask int64 [H,W] in {0,1}), matching the contract expected by everything downstream.
    """
    def __init__(self, image_dir, label_dir, indices=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        all_filenames = sorted(os.listdir(image_dir), key=lambda f: int(os.path.splitext(f)[0]))
        self.filenames = [all_filenames[i] for i in indices] if indices is not None else all_filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = cv2.imread(os.path.join(self.image_dir, fname), cv2.IMREAD_GRAYSCALE)
        label = cv2.imread(os.path.join(self.label_dir, fname), cv2.IMREAD_GRAYSCALE)

        image = image.astype(np.float32) / 255.0
        # In this mirror, label pixels are bright (~255) for cell interior and dark (~0) for
        # membrane/background -- threshold at the midpoint to get a clean binary mask.
        mask = (label > 127).astype(np.int64)

        return image, mask


# Reproducible 24-train / 6-val split of the 30 labeled images (paper trains on all 30 with
# results evaluated by the challenge server; we hold out a slice locally since that server
# is not queryable from this notebook).
_all_filenames = sorted(os.listdir(IMAGE_DIR), key=lambda f: int(os.path.splitext(f)[0]))
_all_indices = list(range(len(_all_filenames)))
random.Random(SEED).shuffle(_all_indices)

N_VAL_RAW = 6
val_indices = sorted(_all_indices[:N_VAL_RAW])
train_indices = sorted(_all_indices[N_VAL_RAW:])
N_TRAIN_RAW = len(train_indices)

print(f"Total labeled images: {len(_all_filenames)} | train: {N_TRAIN_RAW} | val: {N_VAL_RAW}")
print(f"Validation image indices: {val_indices}")


In [ ]:
# ============================================================
# 1.3  Quick look at real samples
# ============================================================
_preview = ISBIMembraneDataset(IMAGE_DIR, LABEL_DIR, indices=train_indices[:3])

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    img, msk = _preview[i]
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(f"EM image (train idx {train_indices[i]})")
    axes[0, i].axis("off")
    axes[1, i].imshow(msk, cmap="viridis")
    axes[1, i].set_title("Ground-truth cell/membrane mask")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()


## 2. Weighted Loss Map — Paper Eq. (2)

$$
w(\mathbf{x}) = w_c(\mathbf{x}) + w_0 \cdot \exp\left(-\frac{(d_1(\mathbf{x}) + d_2(\mathbf{x}))^2}{2\sigma^2}\right)
$$

- $w_c$ balances class frequencies (background vs. foreground pixel counts).
- $d_1(\mathbf{x})$ = distance to the border of the **nearest** cell instance.
- $d_2(\mathbf{x})$ = distance to the border of the **second-nearest** cell instance.
- The paper sets $w_0 = 10$, $\sigma \approx 5$ px.

This term inflates the loss weight of thin **background gaps between touching cells** so the
network is explicitly pushed to learn to separate them — the exact issue shown in Fig. 3 of the paper.
We use `scipy.ndimage.distance_transform_edt` for the Euclidean distance transform and
`scipy.ndimage.label` to get per-instance connected components (since our binary mask only encodes
foreground/background, but Eq. (2) needs *per-instance* borders).


In [ ]:
# ============================================================
# 2.1  Weight-map computation (Eq. 1 & 2 of the paper)
# ============================================================
def compute_class_balance_weight(mask):
    """w_c(x): inverse-frequency class balancing weight map."""
    mask = mask.astype(np.float32)
    n_fg = mask.sum()
    n_bg = mask.size - n_fg
    n_fg = max(n_fg, 1.0)
    n_bg = max(n_bg, 1.0)
    # Normalize so the mean weight over the image is ~1
    w_fg = mask.size / (2.0 * n_fg)
    w_bg = mask.size / (2.0 * n_bg)
    wc = np.where(mask == 1, w_fg, w_bg).astype(np.float32)
    return wc


def compute_boundary_weight(mask, w0=10.0, sigma=5.0):
    """w0 * exp(-(d1+d2)^2 / (2*sigma^2)) using per-instance connected components."""
    labeled, n_instances = cc_label(mask)

    if n_instances < 2:
        # Nothing to separate -> boundary term is zero everywhere
        return np.zeros_like(mask, dtype=np.float32)

    H, W = mask.shape
    # distance_to_instance[k] = distance transform to the border of instance k,
    # evaluated everywhere (positive outside AND inside the instance's own border region
    # is fine here since we only need, for each background pixel, the distance to each
    # instance's boundary; for pixels inside an instance we set self-distance to 0 by
    # excluding that instance's own mask when it's the pixel's own label — matches the
    # paper's intent of measuring how *close* background gaps are to two nearby cells).
    dist_stack = np.zeros((n_instances, H, W), dtype=np.float32)
    for k in range(1, n_instances + 1):
        instance_mask = (labeled == k)
        # distance from every pixel to the boundary of this instance:
        # distance_transform_edt of the *inverse* mask gives distance to nearest True pixel
        dist_stack[k - 1] = distance_transform_edt(~instance_mask)

    dist_sorted = np.sort(dist_stack, axis=0)
    d1 = dist_sorted[0]
    d2 = dist_sorted[1] if n_instances >= 2 else dist_sorted[0]

    boundary_weight = w0 * np.exp(-((d1 + d2) ** 2) / (2.0 * sigma ** 2))
    # Only meaningful in the background (paper: "separating background labels between touching cells")
    boundary_weight = np.where(mask == 0, boundary_weight, 0.0).astype(np.float32)
    return boundary_weight


def compute_weight_map(mask, w0=10.0, sigma=5.0):
    """Full Eq. (2): w(x) = w_c(x) + w0 * exp(-(d1+d2)^2 / (2*sigma^2))."""
    wc = compute_class_balance_weight(mask)
    wb = compute_boundary_weight(mask, w0=w0, sigma=sigma)
    return (wc + wb).astype(np.float32)


In [ ]:
# ============================================================
# 2.2  Visualize weight map on a real sample with touching cells
# ============================================================
_img, _msk = ISBIMembraneDataset(IMAGE_DIR, LABEL_DIR, indices=[train_indices[0]])[0]
_wmap = compute_weight_map(_msk, w0=10.0, sigma=5.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(_img, cmap="gray"); axes[0].set_title("EM image"); axes[0].axis("off")
axes[1].imshow(_msk, cmap="gray"); axes[1].set_title("Ground truth mask"); axes[1].axis("off")
im = axes[2].imshow(_wmap, cmap="jet"); axes[2].set_title("Weight map w(x)  (Eq. 2)"); axes[2].axis("off")
fig.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()


## 3. Data Augmentation — §3.1

The paper stresses that **random elastic deformations on a coarse 3×3 grid**, with displacements
drawn from a Gaussian with **10 px standard deviation** and **bicubic interpolation**, are the key
augmentation for training with very few annotated images — plus shift/rotation invariance and
gray-value robustness.

We implement this with `albumentations.ElasticTransform` (parameterised to match the paper's coarse,
smooth 3×3 grid deformation field) alongside random flips/rotations/shifts/brightness-contrast jitter.
The weight map is **recomputed after augmentation** from the augmented mask, since the whole point
of Eq. (2) is that it depends on the (possibly newly touching) instance geometry.


In [ ]:
# ============================================================
# 3.1  Augmentation pipeline
# ============================================================
def get_train_augmentation(image_size=256):
    return A.Compose([
        A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=25,
                            border_mode=cv2.BORDER_REFLECT, p=0.8),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        # Coarse, smooth elastic deformation approximating the paper's 3x3-grid,
        # sigma~10px displacement field (alpha controls displacement magnitude,
        # alpha_affine adds a mild affine jitter, sigma controls smoothness/coarseness).
        A.ElasticTransform(alpha=34, sigma=10,
                            border_mode=cv2.BORDER_REFLECT, p=0.8),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.GaussNoise(std_range=(0.05, 0.15), p=0.3),
    ])

def get_val_augmentation():
    return A.Compose([])  # identity — no augmentation at validation time


In [ ]:
# ============================================================
# 3.2  U-Net training Dataset wrapper
#      Produces (image_tensor[1,H,W], mask_tensor[H,W], weight_tensor[H,W])
#      `target_size`: if the model uses unpadded convs, the mask/weight map must be
#      CENTER-CROPPED to the network's output resolution (paper: 572 in -> 388 out).
# ============================================================
class UNetSegmentationDataset(Dataset):
    def __init__(self, base_dataset, input_size=256, target_size=256,
                 augmentation=None, w0=10.0, sigma=5.0):
        self.base_dataset = base_dataset
        self.input_size = input_size
        self.target_size = target_size
        self.augmentation = augmentation
        self.w0 = w0
        self.sigma = sigma

    def __len__(self):
        return len(self.base_dataset)

    def _center_crop(self, arr, out_size):
        h, w = arr.shape[:2]
        if h == out_size and w == out_size:
            return arr
        top = (h - out_size) // 2
        left = (w - out_size) // 2
        return arr[top:top + out_size, left:left + out_size]

    def __getitem__(self, idx):
        image, mask = self.base_dataset[idx]

        image = cv2.resize(image, (self.input_size, self.input_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask.astype(np.uint8), (self.input_size, self.input_size),
                           interpolation=cv2.INTER_NEAREST).astype(np.int64)

        if self.augmentation is not None:
            augmented = self.augmentation(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        mask = mask.astype(np.int64)
        weight_map = compute_weight_map(mask, w0=self.w0, sigma=self.sigma)

        # Crop target mask/weight map to the network's output resolution
        mask_t = self._center_crop(mask, self.target_size)
        weight_t = self._center_crop(weight_map, self.target_size)

        image_tensor = torch.from_numpy(image).float().unsqueeze(0)          # [1, H, W]
        mask_tensor = torch.from_numpy(mask_t).long()                        # [H, W]
        weight_tensor = torch.from_numpy(weight_t).float()                   # [H, W]

        return image_tensor, mask_tensor, weight_tensor


print("Augmentation + dataset wrapper ready.")


## 4. Model Architecture — Fig. 1

Two variants are provided:

1. **`UNetOriginal`** — the exact paper architecture: unpadded ("valid") 3×3 convolutions, so the
   spatial size shrinks at every conv. With a 572×572 input this yields a 388×388 output, and skip
   connections must be **center-cropped** to match the decoder feature map size before concatenation
   (paper: *"The cropping is necessary due to the loss of border pixels in every convolution"*).
2. **`UNetPadded`** — a convenience "same"-padding variant (512×512 → 512×512) that keeps
   input/output resolution identical, useful for quick experimentation without needing to
   center-crop targets.

Both share:
- 4 down-sampling stages (2×2 max-pool, stride 2, doubling channels at each stage) and 4
  up-sampling stages (2×2 up-convolution halving channels, concatenation with the cropped
  skip feature map, then two 3×3 convs + ReLU).
- A final 1×1 convolution mapping the 64-channel feature map to `n_classes` output channels.
- 23 convolutional layers total, matching the paper.


In [ ]:
# ============================================================
# 4.1  Building blocks
# ============================================================
class DoubleConv(nn.Module):
    """(3x3 conv -> ReLU) x 2, as used at every U-Net stage (Fig. 1)."""
    def __init__(self, in_ch, out_ch, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=padding),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=padding),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


def center_crop(feature_map, target_size):
    """Center-crop `feature_map` [N,C,H,W] spatially to `target_size` (paper: skip-connection crop)."""
    _, _, h, w = feature_map.shape
    th, tw = target_size
    top = (h - th) // 2
    left = (w - tw) // 2
    return feature_map[:, :, top:top + th, left:left + tw]


In [ ]:
# ============================================================
# 4.2  UNetOriginal — unpadded convolutions, exactly as in the paper (572 -> 388)
# ============================================================
class UNetOriginal(nn.Module):
    """Faithful reproduction of Fig. 1: unpadded 3x3 convs, 4 down/up stages, crop-and-concat skips.

    With the paper's canonical 572x572x1 input this produces a 388x388xn_classes output.
    Any input size works as long as it survives 4 rounds of (2 valid 3x3 convs + 2x2 maxpool)
    without hitting an odd spatial dimension (paper §2: 'select the input tile size such that
    all 2x2 max-pooling operations are applied to a layer with an even x- and y-size').
    """
    def __init__(self, in_channels=1, n_classes=2, base_channels=64):
        super().__init__()
        c = base_channels
        # Contracting path
        self.enc1 = DoubleConv(in_channels, c, padding=0)
        self.enc2 = DoubleConv(c, c * 2, padding=0)
        self.enc3 = DoubleConv(c * 2, c * 4, padding=0)
        self.enc4 = DoubleConv(c * 4, c * 8, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = DoubleConv(c * 8, c * 16, padding=0)

        # Expansive path
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c * 16, c * 8, padding=0)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4, padding=0)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2, padding=0)
        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c * 2, c, padding=0)

        self.out_conv = nn.Conv2d(c, n_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat([center_crop(e4, d4.shape[-2:]), d4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([center_crop(e3, d3.shape[-2:]), d3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([center_crop(e2, d2.shape[-2:]), d2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([center_crop(e1, d1.shape[-2:]), d1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)


In [ ]:
# ============================================================
# 4.3  UNetPadded — convenience "same"-padding variant (512 -> 512)
# ============================================================
class UNetPadded(nn.Module):
    """Same architecture/receptive field structure as UNetOriginal but with padding=1 on every
    3x3 conv, so spatial size is preserved end-to-end (input size == output size). Much easier to
    use for arbitrary image sizes and standard segmentation pipelines.
    """
    def __init__(self, in_channels=1, n_classes=2, base_channels=64):
        super().__init__()
        c = base_channels
        self.enc1 = DoubleConv(in_channels, c, padding=1)
        self.enc2 = DoubleConv(c, c * 2, padding=1)
        self.enc3 = DoubleConv(c * 2, c * 4, padding=1)
        self.enc4 = DoubleConv(c * 4, c * 8, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(c * 8, c * 16, padding=1)

        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c * 16, c * 8, padding=1)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4, padding=1)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2, padding=1)
        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c * 2, c, padding=1)

        self.out_conv = nn.Conv2d(c, n_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([e4, self.up4(b)], dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))

        return self.out_conv(d1)


In [ ]:
# ============================================================
# 4.4  He / Kaiming-normal weight initialization (paper §3)
#      "drawing the initial weights from a Gaussian distribution with a standard
#       deviation of sqrt(2/N), where N is the number of incoming nodes of one neuron."
#      For a 3x3 conv with 64 input channels: N = 9 * 64 = 576  (paper's worked example).
# ============================================================
def weights_init_he(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        # fan_in mode reproduces N = kernel_h * kernel_w * in_channels, std = sqrt(2/N)
        nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)


# Sanity check reproducing the paper's worked example (3x3 conv, 64 in-channels -> N=576)
_example_conv = nn.Conv2d(64, 64, kernel_size=3)
weights_init_he(_example_conv)
_N = 3 * 3 * 64
print(f"Paper example: N = 3*3*64 = {_N}, target std = sqrt(2/N) = {math.sqrt(2/_N):.4f}, "
      f"actual init std = {_example_conv.weight.std().item():.4f}")


In [ ]:
# ============================================================
# 4.5  Instantiate model
#      Choose UNET_VARIANT = 'padded' (recommended for this notebook's 256x256 synthetic data)
#      or 'original' (paper-exact unpadded convs, needs a 572x572-class input, e.g. 260 in -> 68 out
#      for smaller images -- see the size table printed below).
# ============================================================
UNET_VARIANT = "padded"   # "padded" or "original"
IN_CHANNELS = 1
N_CLASSES = 2             # background / cell
BASE_CHANNELS = 64        # paper default (64 at first stage)
IMAGE_SIZE = 256          # working resolution (native EM images are 512x512; downsized for speed —
                          # set to 512 to train at native resolution if your GPU has the headroom)

if UNET_VARIANT == "original":
    # Determine the true output size for the chosen input size by a dry run
    model = UNetOriginal(IN_CHANNELS, N_CLASSES, BASE_CHANNELS).to(device)
    with torch.no_grad():
        _dummy = torch.zeros(1, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE).to(device)
        _out = model(_dummy)
    TARGET_SIZE = _out.shape[-1]
    print(f"UNetOriginal: input {IMAGE_SIZE}x{IMAGE_SIZE} -> output {TARGET_SIZE}x{TARGET_SIZE} "
          f"(paper reference: 572 -> 388)")
else:
    model = UNetPadded(IN_CHANNELS, N_CLASSES, BASE_CHANNELS).to(device)
    TARGET_SIZE = IMAGE_SIZE
    print(f"UNetPadded: input {IMAGE_SIZE}x{IMAGE_SIZE} -> output {TARGET_SIZE}x{TARGET_SIZE} (same padding)")

model.apply(weights_init_he)

n_params = sum(p.numel() for p in model.parameters())
n_conv_layers = sum(1 for m in model.modules() if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)))
print(f"Total parameters       : {n_params:,}")
print(f"Total conv/deconv layers: {n_conv_layers}  (paper: 23 conv layers in the core U shape)")


## 5. Weighted Pixel-Wise Cross-Entropy Loss — Eq. (1)

$$
E = \sum_{\mathbf{x} \in \Omega} w(\mathbf{x}) \cdot \log\big(p_{\ell(\mathbf{x})}(\mathbf{x})\big)
$$

where $p_k(\mathbf{x})$ is the pixel-wise softmax over the $K$-channel output, $\ell(\mathbf{x})$
is the ground-truth class, and $w(\mathbf{x})$ is the weight map from Eq. (2). We implement this as
a per-pixel `NLLLoss` on `log_softmax` outputs, multiplied by the precomputed weight map and averaged.


In [ ]:
# ============================================================
# 5.1  Weighted cross-entropy loss (Eq. 1)
# ============================================================
class WeightedCrossEntropyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, target, weight_map):
        """
        logits:     [N, K, H, W]  raw network output
        target:     [N, H, W]     int64 class indices
        weight_map: [N, H, W]     float per-pixel weight, Eq. (2)
        """
        log_probs = F.log_softmax(logits, dim=1)                      # [N,K,H,W]
        nll = F.nll_loss(log_probs, target, reduction="none")         # [N,H,W]
        weighted = nll * weight_map
        return weighted.sum() / (weight_map.sum() + 1e-8)


criterion = WeightedCrossEntropyLoss()
print("Loss function ready:", criterion)


## 6. Evaluation Metrics

- **IoU / Jaccard Index** — this is exactly what Table 2 of the paper reports for the PhC-U373 /
  DIC-HeLa challenges.
- **Pixel Accuracy** — simple overall correctness, complementary to IoU.


In [ ]:
# ============================================================
# 6.1  IoU (Jaccard) and pixel accuracy
# ============================================================
@torch.no_grad()
def compute_iou(pred, target, n_classes=2, eps=1e-8):
    """Mean IoU over classes. pred, target: [N,H,W] int64."""
    ious = []
    for cls in range(n_classes):
        pred_c = (pred == cls)
        target_c = (target == cls)
        intersection = (pred_c & target_c).sum().float()
        union = (pred_c | target_c).sum().float()
        if union == 0:
            continue
        ious.append((intersection / (union + eps)).item())
    return float(np.mean(ious)) if ious else 0.0


@torch.no_grad()
def compute_pixel_accuracy(pred, target):
    correct = (pred == target).sum().float()
    total = torch.numel(target)
    return (correct / total).item()


## 7. Training Loop

Matches the paper's optimization setup in §3:
- **SGD** with **momentum = 0.99** ("a high momentum such that a large number of the previously
  seen training samples determine the update in the current optimization step").
- **Batch size = 1** by default (paper favors large input tiles over large batch size due to GPU
  memory limits) — configurable via `BATCH_SIZE` since Colab GPUs vary.
- Weight maps are computed per-sample inside the `Dataset` (see §3.2) and passed to the loss.


In [ ]:
# ============================================================
# 7.1  Datasets, splits, and DataLoaders
#      raw_train_ds / raw_val_ds wrap the real ISBI EM images using the train_indices /
#      val_indices split computed in Section 1.2 (24 train / 6 val of the 30 labeled images).
# ============================================================
BATCH_SIZE = 2        # paper uses 1; raise if your GPU has headroom
NUM_WORKERS = 2

raw_train_ds = ISBIMembraneDataset(IMAGE_DIR, LABEL_DIR, indices=train_indices)
raw_val_ds = ISBIMembraneDataset(IMAGE_DIR, LABEL_DIR, indices=val_indices)

train_ds = UNetSegmentationDataset(
    raw_train_ds, input_size=IMAGE_SIZE, target_size=TARGET_SIZE,
    augmentation=get_train_augmentation(IMAGE_SIZE), w0=10.0, sigma=5.0,
)
val_ds = UNetSegmentationDataset(
    raw_val_ds, input_size=IMAGE_SIZE, target_size=TARGET_SIZE,
    augmentation=get_val_augmentation(), w0=10.0, sigma=5.0,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)} | "
      f"Input {IMAGE_SIZE}x{IMAGE_SIZE} -> Target {TARGET_SIZE}x{TARGET_SIZE}")


In [ ]:
# ============================================================
# 7.2  Optimizer (SGD, momentum=0.99 per paper §3) and training config
# ============================================================
LEARNING_RATE = 1e-3
MOMENTUM = 0.99          # paper's high-momentum choice
WEIGHT_DECAY = 0.0
NUM_EPOCHS = 25
CHECKPOINT_PATH = "unet_best.pt"

optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE,
                             momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                         factor=0.5, patience=4)

history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_pixel_acc": []}
best_val_iou = -1.0


In [ ]:
# ============================================================
# 7.3  Train / validate for one epoch
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, masks, weights in loader:
        images = images.to(device)
        masks = masks.to(device)
        weights = weights.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks, weights)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion, device, n_classes=2):
    model.eval()
    running_loss = 0.0
    ious, accs = [], []
    for images, masks, weights in loader:
        images = images.to(device)
        masks = masks.to(device)
        weights = weights.to(device)

        logits = model(images)
        loss = criterion(logits, masks, weights)
        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        ious.append(compute_iou(preds, masks, n_classes=n_classes))
        accs.append(compute_pixel_accuracy(preds, masks))

    return (running_loss / len(loader.dataset),
            float(np.mean(ious)),
            float(np.mean(accs)))


In [ ]:
# ============================================================
# 7.4  Run training
# ============================================================
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_iou, val_acc = validate(model, val_loader, criterion, device, n_classes=N_CLASSES)
    scheduler.step(val_iou)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    history["val_pixel_acc"].append(val_acc)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_iou": val_iou,
            "unet_variant": UNET_VARIANT,
        }, CHECKPOINT_PATH)

    print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"val_IoU={val_iou:.4f} | val_pixelAcc={val_acc:.4f} | "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} min. Best val IoU = {best_val_iou:.4f} "
      f"(checkpoint saved to '{CHECKPOINT_PATH}').")


In [ ]:
# ============================================================
# 7.5  Training curves
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Weighted cross-entropy loss")
axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["val_iou"], color="green")
axes[1].set_title("Validation IoU")
axes[1].set_xlabel("epoch")

axes[2].plot(history["val_pixel_acc"], color="orange")
axes[2].set_title("Validation pixel accuracy")
axes[2].set_xlabel("epoch")

plt.tight_layout()
plt.show()


## 8. Qualitative Visualization

For a handful of validation samples we show, side-by-side: **input image**, **ground-truth mask**,
**weight map** (Eq. 2, highlighting the narrow separating borders between touching cells — compare
to Fig. 3d of the paper), and the **network's predicted segmentation**.


In [ ]:
# ============================================================
# 8.1  Load best checkpoint and visualize predictions
# ============================================================
_ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(_ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {_ckpt['epoch']} (val IoU = {_ckpt['val_iou']:.4f})")

N_VIS = 4
fig, axes = plt.subplots(N_VIS, 4, figsize=(16, 4 * N_VIS))

with torch.no_grad():
    for i in range(N_VIS):
        image, mask, weight = val_ds[i]
        logits = model(image.unsqueeze(0).to(device))
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

        img_np = image.squeeze(0).numpy()
        mask_np = mask.numpy()
        weight_np = weight.numpy()

        axes[i, 0].imshow(img_np, cmap="gray");    axes[i, 0].set_title("Input image");      axes[i, 0].axis("off")
        axes[i, 1].imshow(mask_np, cmap="gray");    axes[i, 1].set_title("Ground truth");     axes[i, 1].axis("off")
        im = axes[i, 2].imshow(weight_np, cmap="jet"); axes[i, 2].set_title("Weight map w(x)");   axes[i, 2].axis("off")
        axes[i, 3].imshow(pred, cmap="gray");       axes[i, 3].set_title("Predicted mask");   axes[i, 3].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 8.2  Per-sample quantitative summary on the full validation set
# ============================================================
model.eval()
sample_ious, sample_accs = [], []
with torch.no_grad():
    for images, masks, weights in val_loader:
        images, masks = images.to(device), masks.to(device)
        preds = torch.argmax(model(images), dim=1)
        sample_ious.append(compute_iou(preds, masks, n_classes=N_CLASSES))
        sample_accs.append(compute_pixel_accuracy(preds, masks))

print(f"Validation set  (n={len(sample_ious)})")
print(f"  Mean IoU            : {np.mean(sample_ious):.4f}  (std {np.std(sample_ious):.4f})")
print(f"  Mean pixel accuracy : {np.mean(sample_accs):.4f}  (std {np.std(sample_accs):.4f})")


## 9. Using a Different Real Dataset

This notebook trains on the real **ISBI 2012 EM segmentation challenge** data by default (Section 1) —
the same dataset the paper's Table 1 results are reported on. To switch to a different real
biomedical (or general) segmentation dataset instead:

1. **Kaggle 2018 Data Science Bowl** (nuclei segmentation, closest in spirit to the paper's
   touching-cell problem, but with per-image instance masks rather than one shared cell/membrane mask):
   ```python
   # !pip install -q kaggle
   # from google.colab import files; files.upload()  # upload kaggle.json API token
   # !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
   # !kaggle competitions download -c data-science-bowl-2018
   ```
2. **PhC-U373 / DIC-HeLa** (ISBI cell-tracking challenge 2015, Table 2 of the paper) —
   registration required at the official [Cell Tracking Challenge](http://celltrackingchallenge.net/) site.
3. **Oxford-IIIT Pet** (binary/trimap segmentation, no registration, available directly via
   `torchvision.datasets.OxfordIIITPet(..., target_types="segmentation")`) — a convenient sanity
   check of the pipeline on natural (non-biomedical) images.

In every case, only `ISBIMembraneDataset` needs to be replaced — write a `Dataset.__getitem__` that
returns `(image[H,W] float32 in [0,1], mask[H,W] int64 in {0,1})`, or a mask with per-instance labels
if you want per-instance IDs for the weight map (`compute_weight_map` will treat a plain 0/1 mask via
connected components, as done above, if instance IDs aren't available). Everything else
(augmentation, weight maps, loss, model, training loop, metrics, visualization) works unchanged.
